# Receipt OCR — Responses API + structured outputs (Pydantic)

**Pydantic models and prompts live in this notebook** so you can experiment without touching the backend. When you are happy with the shape, mirror changes into **`backend/app/schemas/ocr.py`**. The app still uses Chat Completions in **`app/services/ocr.py`**; this notebook uses the **[Responses API](https://platform.openai.com/docs/api-reference/responses/create)** (`client.responses.parse`).

**Docs:** [Create a response](https://platform.openai.com/docs/api-reference/responses/create) · [Migrate from Chat Completions](https://platform.openai.com/docs/guides/migrate-to-responses) · [Vision / image inputs](https://platform.openai.com/docs/guides/images) · [SDK example](https://github.com/openai/openai-python/blob/main/examples/responses/structured_outputs.py)

```bash
pip install -U "openai>=1.76" pydantic python-dotenv jupyter pillow pillow-heif
jupyter notebook receipt_ocr_openai_test.ipynb
```

**HEIC (iPhone):** Pillow needs **`pillow-heif`** (see above). On macOS, if that import fails, the helper below tries **`sips`** to convert to JPEG.

In [1]:
from __future__ import annotations

import os
from decimal import Decimal
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, model_validator

ROOT = Path.cwd()

class OCRLineItemExtracted(BaseModel):
    """Single line item with forced non-null values for consistency."""
    model_config = ConfigDict(extra="forbid")

    description: str = Field(..., description="Item label as printed on the receipt.")
    
    # Force quantity to int and default to 1
    quantity: int = Field(default=1, description="The count or weight. Default to 1 if not specified.")
    
    # Ensure all money fields are strings and always populated
    price_per_unit: str = Field(..., description="The price of a single unit. If not listed, use line_subtotal.")
    line_subtotal: str = Field(..., description="The amount printed next to the item name (before discounts).")
    line_discount: str = Field(default="0.00", description="The discount amount. Default to '0.00' if no discount exists.")
    final_line_total: str = Field(..., description="The final cost for this line: (price_per_unit * quantity) - line_discount.")
    
    confidence: float = Field(0.0, description="Confidence score in the range [0, 1]")

class ReceiptOCRResult(BaseModel):
    """Structured receipt parse with non-null totals."""
    model_config = ConfigDict(extra="forbid")

    merchant_name: str = Field("Unknown", description="Store name. Default to 'Unknown'.")
    currency: str = Field("ZAR", description="Currency code. Default to 'ZAR'.")
    total_amount: str = Field(..., description="Grand total (Gross amount).")
    items: list[OCRLineItemExtracted] = Field(default_factory=list)

    @model_validator(mode="after")
    def synthetic_total_line(self) -> ReceiptOCRResult:
        if self.items or self.total_amount is None:
            return self
        amt = str(Decimal(str(self.total_amount)).quantize(Decimal("0.01")))
        return self.model_copy(
            update={
                "items": [
                    OCRLineItemExtracted(
                        description=self.merchant_name,
                        price_per_unit=amt,
                        line_subtotal=amt,
                        final_line_total=amt,
                        quantity=1,
                        line_discount="0.00",
                        confidence=1.0
                    )
                ]
            }
        )

# --- Optimized Prompts ---

SYSTEM_PROMPT = """You are a specialized vision-to-structured-data assistant for South African retail receipts.
Your goal is to provide a clean JSON where EVERY numeric field is populated (no nulls).

### Extraction Logic:
1. QUANTITY & PRICE: If an item does not have an '@' multiplier line, set 'quantity' to 1 and 'price_per_unit' to match the 'line_subtotal'.
2. DISCOUNTS: Lines like 'XTRASAVE' or negative values are discounts. Attach them to the preceding item. If no discount exists, set 'line_discount' to "0.00".
3. LINE TOTALS: 'final_line_total' MUST equal ('line_subtotal' minus 'line_discount').
4. MATH CONSISTENCY: Ensure 'line_subtotal' equals ('price_per_unit' * 'quantity').
5. MULTIPLE IMAGES: The same physical receipt line may appear in more than one image (e.g. overlap between photos, folds, or stitching). **Do not duplicate** items: merge or choose a single canonical entry per line; each purchased line must appear at most once in `items`.

### Example Transformation:
Input: 
T/BRUSH WHT MED       R89.98
            2 @       R44.99
XTRASAVE DENTAL      -R27.00

Output Item:
{
  "description": "T/BRUSH WHT MED",
  "quantity": 2,
  "price_per_unit": "44.99",
  "line_subtotal": "89.98",
  "line_discount": "27.00",
  "final_line_total": "62.98"
}

Input:
FEATHERLITE CON 12 V  R240.00

Output Item:
{
  "description": "FEATHERLITE CON 12 V",
  "quantity": 1,
  "price_per_unit": "240.00",
  "line_subtotal": "240.00",
  "line_discount": "0.00",
  "final_line_total": "240.00"
}
"""

USER_PROMPT = """
Extract all line items and the grand total (GROSS) from this receipt.
- Ensure 'quantity' is an integer.
- Ensure 'line_discount' is "0.00" if no discount is present.
- Do not return null for any numeric fields; infer the unit price if it is not explicitly stated.
- Ensure the 'merchant_name' is extracted (e.g., 'Checkers Rondebosch' or 'Clicks').
"""

USER_PROMPT_MULTI_PAGE = """
Multiple images are attached: treat them as sequential parts of ONE receipt (e.g. multiple pages or separate photos of the same slip). Combine all line items in reading order; use the correct grand total (usually on the page that shows the payment/total summary).
"""

MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o")

print("Models:", OCRLineItemExtracted.__name__, ",", ReceiptOCRResult.__name__)
print("MODEL:", MODEL)

Models: OCRLineItemExtracted , ReceiptOCRResult
MODEL: gpt-5.2


In [2]:
import base64
import sys
from collections.abc import Sequence

from openai import OpenAI


def image_file_to_jpeg_base64(path: str | Path) -> str:
    """JPEG bytes as base64 for `data:image/jpeg;base64,...`. Handles HEIC via pillow-heif or macOS sips."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in (".jpg", ".jpeg"):
        return base64.b64encode(path.read_bytes()).decode()

    from io import BytesIO

    if suffix in (".heic", ".heif"):
        try:
            from pillow_heif import register_heif_opener

            register_heif_opener()
        except ImportError:
            if sys.platform == "darwin":
                import subprocess
                import tempfile

                with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                    out_path = tmp.name
                try:
                    cp = subprocess.run(
                        [
                            "sips",
                            "-s",
                            "format",
                            "jpeg",
                            str(path.resolve()),
                            "--out",
                            out_path,
                        ],
                        check=False,
                        capture_output=True,
                        text=True,
                    )
                    if cp.returncode != 0:
                        err = (cp.stderr or cp.stdout or "").strip()
                        raise RuntimeError(
                            f"sips could not convert HEIC (exit {cp.returncode}). "
                            f"Try: pip install pillow-heif. {err}"
                        )
                    return base64.b64encode(Path(out_path).read_bytes()).decode()
                finally:
                    Path(out_path).unlink(missing_ok=True)
            raise RuntimeError(
                "HEIC/HEIF needs pillow-heif: pip install pillow-heif "
                "(on macOS without it, sips is tried automatically)."
            ) from None

    try:
        from PIL import Image
    except ImportError:
        raise RuntimeError(
            "Install pillow to convert non-JPEG images, or use a .jpg file."
        ) from None

    raw = path.read_bytes()
    try:
        img = Image.open(BytesIO(raw)).convert("RGB")
    except Exception as e:
        raise RuntimeError(
            f"Could not decode {path.name!r}. For iPhone HEIC: pip install pillow-heif"
        ) from e
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode()


def image_paths_to_jpeg_base64_list(paths: Sequence[str | Path]) -> list[str]:
    """Encode each path to JPEG base64 (HEIC and JPG can be mixed in one list)."""
    return [image_file_to_jpeg_base64(p) for p in paths]


def _parsed_receipt_from_response(rsp) -> ReceiptOCRResult:
    """Walk Responses output; matches SDK structured output example."""
    for output in rsp.output:
        if output.type != "message":
            continue
        for item in output.content:
            if item.type != "output_text":
                continue
            if item.parsed is not None:
                return item.parsed
    raise RuntimeError(f"No parsed {ReceiptOCRResult.__name__} in response output: {rsp.output!r}")


def extract_receipt_items_sync(
    image_base64: str | list[str],
    api_key: str,
    model: str | None = None,
    *,
    client: OpenAI | None = None,
) -> ReceiptOCRResult:
    """Vision + structured parse via Responses API (`/v1/responses`). Pass one base64 string or a list (multi-page)."""
    if isinstance(image_base64, str):
        images: list[str] = [image_base64]
    else:
        images = list(image_base64)
    if not images:
        raise ValueError("At least one JPEG base64 string is required (non-empty list).")

    prompt = USER_PROMPT.strip()
    if len(images) > 1:
        prompt = prompt + "\n\n" + USER_PROMPT_MULTI_PAGE.strip()

    content: list[dict] = [{"type": "input_text", "text": prompt}]
    for b64 in images:
        content.append(
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{b64}",
                "detail": "high",
            }
        )

    c = client or OpenAI(api_key=api_key, timeout=120.0)
    m = model or MODEL
    m = "gpt-5.2"
    rsp = c.responses.parse(
        model=m,
        instructions=SYSTEM_PROMPT,
        input=[{"role": "user", "content": content}],
        text_format=ReceiptOCRResult,
        temperature=0,
    )
    return _parsed_receipt_from_response(rsp)


print("Ready: OpenAI().responses.parse(..., text_format=ReceiptOCRResult)")

Ready: OpenAI().responses.parse(..., text_format=ReceiptOCRResult)


## API key

Loads **`.env`** from the current working directory (typically the repo root). You can also use the **`OPENAI_API_KEY`** environment variable or pass **`api_key`** into **`OpenAI(...)`**.

In [3]:
try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not API_KEY:
    print("Set OPENAI_API_KEY in the environment or .env")
else:
    print("API key loaded (length)", len(API_KEY))

API key loaded (length) 164


## Run on image(s)

Set **`IMAGE_PATHS`** to one or more files. **HEIC** and **JPG** can be mixed; each path is converted to JPEG before the API call.

In [4]:
IMAGE_PATHS: list[Path] = [
    Path("/Users/andrewallkin/Downloads/IMG_5894.HEIC"),
    # Path("/Users/andrewallkin/Downloads/7fc0a06f-ef04-4819-8266-210e2b4b6ed1.JPG"),
    # Path("/Users/andrewallkin/Downloads/3e85e090-a6b7-424e-abdd-426406d03829.JPG")
]

if not API_KEY:
    print("Skip: no API key")
elif not IMAGE_PATHS:
    print("Skip: IMAGE_PATHS is empty — add at least one Path")
elif all(p.exists() for p in IMAGE_PATHS):
    b64_list = image_paths_to_jpeg_base64_list(IMAGE_PATHS)
    result = extract_receipt_items_sync(b64_list, API_KEY)
    print(result.model_dump_json(indent=2))
else:
    missing = [p for p in IMAGE_PATHS if not p.exists()]
    print("Skip: file(s) not found:", [str(p.resolve()) for p in missing])

{
  "merchant_name": "ST HELENA BAY SPAR & TOPS",
  "currency": "ZAR",
  "total_amount": "265.31",
  "items": [
    {
      "description": "F/L PLUM TOMS 200GR",
      "quantity": 1,
      "price_per_unit": "29.99",
      "line_subtotal": "29.99",
      "line_discount": "0.00",
      "final_line_total": "29.99",
      "confidence": 0.93
    },
    {
      "description": "I/PAARMAN'S R/MARY O 200ML",
      "quantity": 1,
      "price_per_unit": "44.99",
      "line_subtotal": "44.99",
      "line_discount": "0.00",
      "final_line_total": "44.99",
      "confidence": 0.9
    },
    {
      "description": "LAMB LOIN CHOPS.. KG",
      "quantity": 1,
      "price_per_unit": "135.60",
      "line_subtotal": "135.60",
      "line_discount": "0.00",
      "final_line_total": "135.60",
      "confidence": 0.88
    },
    {
      "description": "G/L SPONGES 2'S",
      "quantity": 1,
      "price_per_unit": "19.99",
      "line_subtotal": "19.99",
      "line_discount": "0.00",
      "final_